In [ ]:
%matplotlib inline

In [ ]:
import matplotlib.image
import matplotlib.pyplot as plt
from mpltools import annotation

# Session 2: Further Differentiable Programming with Autograd and JAX

<div class="alert alert-block alert-warning" style="background-color: rgb(236,176,146); border: 2px solid rgb(213,104,79); color: rgb(64,64,64);">
    
<b>Set up codespace now. (It will take a while!)</b>

We will show you where to find this notebook in the repo while you wait.

</div>

## Learning objectives

In today's session we will:

* Learn about forward and reverse modes applied to functions of several variables.
* Calculate higher order derivatives using Autograd.
* Try out the JAX differentiable programming framework and see showcases of more advanced AD usage.
* Learn about checkpointing and using AD to compute higher order derivatives.

## Recall: Forward and reverse mode for scalar functions

In session 1, given three scalar functions $f$, $g$, and $h$, which may be composed as
$$\ell(x)=h(g(f(x)),$$
we presented forward mode as evaluating from right-to-left:
$$\frac{\mathrm{d}\ell}{\mathrm{d}x}=\frac{\mathrm{d}h}{\mathrm{d}g}\left(\frac{\mathrm{d}g}{\mathrm{d}f}\frac{\mathrm{d}f}{\mathrm{d}x}\right)$$
and reverse mode as evaluating from left-to-right:
$$\frac{\mathrm{d}\ell}{\mathrm{d}x}=\left(\frac{\mathrm{d}h}{\mathrm{d}g}\frac{\mathrm{d}g}{\mathrm{d}f}\right)\frac{\mathrm{d}f}{\mathrm{d}x}.$$

## Forward mode for vector functions

Suppose we have a function mapping between vectors, $\mathbf{f}:A\rightarrow\mathbb{R}^n$, where $A\subseteq\mathbb{R}^m$, and a point $\mathbf{x}\in A$ at which we seek to evaluate its derivative.

<div class="alert alert-block alert-warning" style="background-color: rgb(236,176,146); border: 2px solid rgb(213,104,79); color: rgb(64,64,64);">
<b>Note</b>
We can interpret such a function as having one or more scalar inputs and one or more scalar outputs.
</div>

For a general definition of forward mode, we need to consider a *seed vector*, $\dot{\mathbf{x}}\in\mathbb{R}^m$. Forward mode allows us to compute the *action* (matrix-vector product)
$$\text{JVP}(\mathbf{f},\mathbf{x},\dot{\mathbf{x}}):=\nabla\mathbf{f}(\mathbf{x})\,\dot{\mathbf{x}}.$$
Here $\nabla\mathbf{f}$ is referred to as the *Jacobian* for the map, so the above is known as a *Jacobian-vector product (JVP)*.
You might also hear the related term *tangent linear model (TLM)*.

Think of the seed vector as the direction in which we want to compute the derivative.
In practice, the seed vector is often a derivative of some upstream code from outside of the part of the program being differentiated.
That is, the upstream code is *passive*, whereas the part we are interested in is *active*, as far as AD is concerned.

<div class="alert alert-block alert-warning" style="background-color: rgb(236,176,146); border: 2px solid rgb(213,104,79); color: rgb(64,64,64);">
<b>Note</b>
The computation is <em>matrix-free</em>. We don't actually need to assemble the Jacobian when we compute this product.
</div>

<div class="alert alert-block alert-info" style="background-color: rgb(195,223,220); border: 2px solid rgb(157,204,199); color: rgb(0,100,100);">
<b>Question</b>
    
In the scalar case we assumed a value for the seed. What was it?

</div>

## Chaining forward mode derivatives

But how does this correspond to the case with three functions?

Suppose in addition to $\mathbf{f}:A\rightarrow\mathbb{R}^n$ with $A\subseteq\mathbb{R}^m$ we have $\mathbf{g}:B\rightarrow\mathbb{R}^k$ with $B\subseteq\mathbb{R}^m$, $\mathbf{h}:C\rightarrow\mathbb{R}^r$ with $C\subseteq\mathbb{R}^k$, and their composition $\boldsymbol{\ell}=\mathbf{f}\circ\mathbf{g}\circ\mathbf{h}$, where the notation here implies
$$
\mathbf{f}\circ\mathbf{g}\circ\mathbf{h}(\mathbf{x})=\mathbf{h}(\mathbf{g}(\mathbf{f}(\mathbf{x})))
$$
for $\mathbf{x}\in A$. Then
$$
\begin{align}
\mathrm{JVP}(\boldsymbol{\ell},\mathbf{x},\dot{\mathbf{x}})
&=\mathrm{JVP}(\mathbf{f}\circ\mathbf{g}\circ\mathbf{h},\,\mathbf{x},\,\dot{\mathbf{x}})\\
&=\nabla(\mathbf{f}\circ\mathbf{g}\circ\mathbf{h})(\mathbf{x})\,\dot{\mathbf{x}}\\
&=\nabla(\mathbf{g}\circ\mathbf{h})(\mathbf{f}(\mathbf{x}))\,\left(\nabla\mathbf{f}(\mathbf{x})\,\dot{\mathbf{x}}\right)\\
&=\nabla\mathbf{h}(\mathbf{g}(\mathbf{f}(\mathbf{x}))\,\left(\nabla\mathbf{g}(\mathbf{f}(\mathbf{x}))\,\left(\nabla\mathbf{f}(\mathbf{x})\,\dot{\mathbf{x}}\right)\right)\\
&=\mathrm{JVP}(\mathbf{h},\,\mathbf{g}(\mathbf{f}(\mathbf{x})),\,\mathrm{JVP}(\mathbf{g},\,\mathbf{f}(\mathbf{x}),\,\mathrm{JVP}(\mathbf{f},\mathbf{x},\dot{\mathbf{x}})))
\end{align}
$$
by two applications of the chain rule. So if we can compute a function and evaluate its forward mode derivative at the same time then we can compute such compositions and their derivatives efficiently, too.

## Reverse mode for vector functions

Consider the same vector function as above, $\mathbf{f}:A\rightarrow\mathbb{R}^n$, where $A\subseteq\mathbb{R}^m$, and a point $\mathbf{x}\in A$.

Given $\mathbf{x}\in A$ and a seed vector $\bar{\mathbf{y}}\in\mathbb{R}^m$, reverse mode AD allows us to compute the *transpose action* (transposed matrix-vector product)
$$\text{JTVP}(\mathbf{f},\mathbf{x},\bar{\mathbf{y}}):=\nabla\mathbf{f}(\mathbf{x})^T\bar{\mathbf{y}}.$$

<div class="alert alert-block alert-warning" style="background-color: rgb(236,176,146); border: 2px solid rgb(213,104,79); color: rgb(64,64,64);">
    
<b>Notes:</b>

- The dimension of the seed vector corresponds to that of the output rather than the input.
- Again, the computation is <em>matrix-free</em>. We don't actually need the Jacobian or its transpose when we compute this product.
- Our original definition for the scalar case assumed the same seed as above.

</div>

<div class="alert alert-block alert-info" style="background-color: rgb(195,223,220); border: 2px solid rgb(157,204,199); color: rgb(0,100,100);">

<b>Optional exercises</b>

1. Convince yourself that the JTVP is well defined.
2. Repeat the above exercise to convice yourself that the definition coincides with the scalar definition, i.e.

$$
\mathrm{JTVP}(\boldsymbol{\ell},\mathbf{x},\bar{\mathbf{y}})
=\mathrm{JTVP}(\mathbf{f},\,\mathbf{x},\,\mathrm{JTVP}(\mathbf{g},\,\mathbf{f}(\mathbf{x}),\,\mathrm{JTVP}(\mathbf{h},\mathbf{g}(\mathbf{f}(\mathbf{x})),\bar{\mathbf{y}}))).
$$

<b>Solution 1</b>

<details>

We have $\nabla\mathbf{f}(\mathbf{x})\in\mathbb{R}^{m\times n}$, so $\nabla\mathbf{f}(\mathbf{x})^T\in\mathbb{R}^{n\times m}$. Since $\bar{\mathbf{y}}\in\mathbb{R}^m$, the dimensions are appropriate to take the JTVP.

</details>

<br>
<b>Solution 2</b>

<details>

$$
\begin{align}
\mathrm{JTVP}(\boldsymbol{\ell},\mathbf{x},\bar{\mathbf{y}})
&=\mathrm{JVP}(\mathbf{f}\circ\mathbf{g}\circ\mathbf{h},\,\mathbf{x},\,\bar{\mathbf{y}})\\
&=\nabla(\mathbf{f}\circ\mathbf{g}\circ\mathbf{h})(\mathbf{x})^T\,\bar{\mathbf{y}}\\
&=\left(\nabla\mathbf{h}(\mathbf{g}(\mathbf{f}(\mathbf{x}))\,\nabla\mathbf{g}(\mathbf{f}(\mathbf{x}))\,\nabla\mathbf{f}(\mathbf{x})\right)^T\bar{\mathbf{y}}\\
&=\left(\nabla\mathbf{f}(\mathbf{x})^T\,\nabla\mathbf{g}(\mathbf{f}(\mathbf{x}))^T\,\nabla\mathbf{h}(\mathbf{g}(\mathbf{f}(\mathbf{x}))^T\right)\,\bar{\mathbf{y}}\\
&=\nabla\mathbf{f}(\mathbf{x})^T\,\left(\nabla\mathbf{g}(\mathbf{f}(\mathbf{x}))^T\,\left(\nabla\mathbf{h}(\mathbf{g}(\mathbf{f}(\mathbf{x}))^T\,\bar{\mathbf{y}}\right)\right)\\
&=\mathrm{JTVP}(\mathbf{f},\,\mathbf{x},\,\mathrm{JTVP}(\mathbf{g},\,\mathbf{f}(\mathbf{x}),\,\mathrm{JTVP}(\mathbf{h},\mathbf{g}(\mathbf{f}(\mathbf{x})),\bar{\mathbf{y}})))
\end{align}
$$

where we have used the result from before and applied the rule of transposes for matrix multiplication.

</details>


</div>

## Forward mode vs. reverse mode

For seed vectors $\dot{\mathbf{x}}\in\mathbb{R}^n$ and $\bar{\mathbf{y}}\in\mathbb{R}^m$, forward mode and reverse mode compute

$$
    \text{JVP}(\mathbf{f},\mathbf{x},\dot{\mathbf{x}}):=\nabla\mathbf{f}(\mathbf{x})\dot{\mathbf{x}}
    \quad\text{and}\quad
    \text{JTVP}(\mathbf{f},\mathbf{x},\bar{\mathbf{y}}):=\nabla\mathbf{f}(\mathbf{x})^T\bar{\mathbf{y}}
$$
respectively.

* Forward mode is more appropriate if $n\ll m$, i.e., $\#inputs\ll\#outputs$.
  * e.g., sensitivity analysis or optimisation w.r.t. a small number of parameters.
* Reverse mode is more appropriate if $n\gg m$, i.e., $\#inputs\gg\#outputs$.
  * e.g., ODE/PDE-constrained optimisation (cost function), machine learning training (loss function), goal-oriented error estimation (quantity of interest).
* Forward mode is computed *eagerly*, whereas reverse mode is done separately from the primal run.
* Reverse mode tends to have higher memory requirements.

<div class="alert alert-block alert-danger"; border: 2px solid rgb(213,104,79); color: rgb(64,64,64);">
    
TODO: Material on higher derivatives ([#22](https://github.com/Cambridge-ICCS/differentiable-programming-summer-school-2026/issues/22))

</div>

<div class="alert alert-block alert-danger"; border: 2px solid rgb(213,104,79); color: rgb(64,64,64);">
    
TODO: Mention source transformation ([#26](https://github.com/Cambridge-ICCS/differentiable-programming-summer-school-2026/issues/26))

</div>

<div class="alert alert-block alert-danger"; border: 2px solid rgb(213,104,79); color: rgb(64,64,64);">
    
TODO: Simple example in JAX ([#24](https://github.com/Cambridge-ICCS/differentiable-programming-summer-school-2026/issues/24))

</div>

<div class="alert alert-block alert-danger"; border: 2px solid rgb(213,104,79); color: rgb(64,64,64);">
    
TODO: Pitch mini-project ([#23](https://github.com/Cambridge-ICCS/differentiable-programming-summer-school-2026/issues/23))

</div>

## Summary and outlook

In today’s session we:

* Learnt about forward and reverse modes applied to functions of several variables.
* Calculated higher order derivatives using Autograd.
* Tried out the JAX differentiable programming framework and see showcases of more advanced AD usage.
* Learnt about checkpointing and using AD to compute higher order derivatives.

## European workshop on Automatic differentiation

<div style="text-align: center;">
  <img src="https://cambridge-iccs.github.io/euroad29/_images/cms.png" width="600" style="display: block; margin-left: auto; margin-right: auto;"/>
  <strong>Figure 1:</strong> Centre of Mathematical Sciences, University of Cambridge</a>.
</div>

ICCS will be hosting the 29th European workshop on Automatic Differentiation (EuroAD) in Cambridge on the **29th-30th September 2026**. It will be an informal meeting of researchers and software engineers who develop and apply automatic differentiation theory and software.

Attendees from all career stages are welcome, particularly PhD students and early career researchers and software engineers.

We welcome contributions with both theoretical and practical perspectives. A selection of possible topics include: new methods for AD, software developments and inter-comparison, AD in machine learning, and applications in science, engineering, and beyond.

Register at
https://cambridge-iccs.github.io/euroad29/

## References

* R. E. Wengert. *A simple automatic derivative evaluation program* (1964). Communications
of the ACM, 7(8):463–464, [doi.org:10.1145/355586.364791](https://doi.org/10.1145/355586.364791).
* S. Linnainmaa. *Taylor expansion of the accumulated rounding error*. BIT,
16(2):146–160, 1976, [doi:10.1007/BF01931367](https://doi.org/10.1007/BF01931367).
* B. Speelpenning. *Compiling fast partial derivatives of functions given by algorithms*.
University of Illinois, 1980, [doi:10.2172/5254402](https://doi.org/10.2172/5254402).
* A. Griewank. *Achieving logarithmic growth of temporal and spatial complexity in
reverse automatic differentiation.* Optimization Methods & Software, 1:35–54, 1992, [doi:10.1080/10556789208805505](https://doi.org/10.1080/10556789208805505).